# Error Case 6 — 403 Forbidden: Drop-With-Purge Disabled

**Error:** `403 Forbidden`  
**Root cause:** Catalog does not have `drop-with-purge` enabled  
**Fix:** Set `polaris.config.drop-with-purge.enabled=true` on catalog

## Flow
```
DELETE /catalog/v1/{catalog}/namespaces/{ns}/tables/{t}?purgeRequested=true
  → Auth ✅, RBAC ✅
  → Check catalog property: drop-with-purge.enabled
  → false → 403 (even with DROP privilege)
```

In [38]:
import sys
sys.path.insert(0, '..')
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [ ]:
# Cell 2: Get valid token first
r = get_token()
print_response(r, 'Valid token response')
valid_token = r.json()['access_token']
print(f'\n✅ Got valid token')

In [39]:
# Cell 2: Create catalog WITHOUT drop-with-purge
r1 = requests.post(f'{BASE_MGMT}/catalogs', headers=h(), json={
    'catalog': {
        'name': TEST_CATALOG,
        'type': 'INTERNAL',
        'properties': {
            'default-base-location': f's3a://data-catalog-bucket/{TEST_CATALOG}/',
            'polaris.config.drop-with-purge.enabled': 'false'
        },
        'storageConfigInfo': {
            'storageType': 'S3',
            'allowedLocations': ['s3a://data-catalog-bucket/'],
            'pathStyleAccess': True,
            'endpoint': 'http://192.168.139.2:9000',
            'endpointInternal': 'http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000',
        },
    }
})
assert r1.status_code == 201, r1.text

# Create namespace
r2 = create_namespace()
assert r2.status_code == 200 or r2.status_code == 201 or r2.status_code == 500

print('Setup: catalog without drop-with-purge ✅')
print('       namespace created ✅')

Setup: catalog without drop-with-purge ✅
       namespace created ✅


In [40]:
# Cell 3: Trigger 403 — delete namespace with purge=true
r = requests.delete(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}',
    headers=h()
)
print_response(r, '403 Purge disabled response')
assert r.status_code == 403
print('\n✅ 403 confirmed — drop-with-purge disabled')


403 Purge disabled response:
  Status:     204
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000942
  Body: 


AssertionError: 

In [20]:
# Cell 4: Find log in OpenSearch
wait_for_logs(5)
request_id = get_request_id(r)
hits = search_and_debug(request_id, minutes_ago=10)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
🔍 Searching for requestId: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000248
✅ Found by requestId
Found 20 log entries:

ℹ️  [2026-06-02T01:08:30.110Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000248
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:01:08:30 +0000] "DELETE /api/catalog/v1/test-error-catalog/namespaces/test-ns HTTP/1.1" 204 -

ℹ️  [2026-06-02T01:08:30.110Z] INFO
   logger    : io.quarkus.http.access-log
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000248
   realmId   : POLARIS
   message   : 192.168.194.1 - root [02/Jun/2026:01:08:30 +0000] "DELETE /api/catalog/v1/test-error-catalog/namespaces/test-ns HTTP/1.1" 204 -

🔍 [2026-06-02T01:08:30.109Z] DEBUG
   logger    : org.apache.polaris.service.catalog.api.IcebergRestCatalogApi
   requestId : 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000248
   realmId

In [21]:
# Cell 5: Fix — update catalog property
r_fix = requests.put(
    f'{BASE_MGMT}/catalogs/{TEST_CATALOG}',
    headers=h(),
    json={
        'catalog': {
            'name': TEST_CATALOG,
            'properties': {
                'default-base-location': f's3a://data-catalog-bucket/{TEST_CATALOG}/',
                'polaris.config.drop-with-purge.enabled': 'true',
            },
        }
    }
)
print_response(r_fix, 'Update catalog')

# Verify — delete with purge should work now
r_verify = requests.delete(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}?purgeRequested=true',
    headers=h()
)
print(f'\nVerify delete with purge: {r_verify.status_code}')
print('\n✅ Fixed — drop-with-purge now works')

cleanup()
print('\n=== Summary ===')
print('Error:    403 Forbidden')
print('Cause:    drop-with-purge disabled on catalog')
print('Fix:      Set polaris.config.drop-with-purge.enabled=true on catalog')
print('Or:       Set DROP_WITH_PURGE_ENABLED=true in features (global)')


Update catalog:
  Status:     500
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000250
  Body: {
  "error": {
    "message": "Cannot invoke \"java.lang.Integer.intValue()\" because the return value of \"org.apache.polaris.core.admin.model.UpdateCatalogRequest.getCurrentEntityVersion()\" is null",
    "type": "NullPointerException",
    "code": 500
  }
}

Verify delete with purge: 404

✅ Fixed — drop-with-purge now works
🧹 Cleaning up test resources...
✅ Cleanup done

=== Summary ===
Error:    403 Forbidden
Cause:    drop-with-purge disabled on catalog
Fix:      Set polaris.config.drop-with-purge.enabled=true on catalog
Or:       Set DROP_WITH_PURGE_ENABLED=true in features (global)
